In [1]:
import sys
sys.path.append("..")   # add main_folder to path


from __future__ import annotations
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, Iterable
from scipy.stats import gaussian_kde, genextreme
from scipy.ndimage import gaussian_filter
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path
import tomllib
import pyarrow.dataset as pds
import pyarrow.compute as pc
import pyarrow as pa
from cartopy.feature import NaturalEarthFeature
import math

from src.genesis.genesis_utils import read_ibtracs, preprocess_ibtracs

In [2]:
#Helpers
def _wrap_lon(lon: pd.Series | np.ndarray) -> np.ndarray:
    """Wrap longitudes to [-180, 180]."""
    arr = np.asarray(lon, dtype=float)
    arr = ((arr + 180.0) % 360.0) - 180.0
    # Force exactly 180 to be -180 for consistent binning
    arr[arr == 180.0] = -180.0
    return arr

def _as_datetime(x: pd.Series, utc: bool = False) -> pd.Series:
    s = pd.to_datetime(x, errors="coerce")
    if utc:
        try:
            s = s.dt.tz_convert("UTC")  # if timezone-aware
        except Exception:
            s = s.dt.tz_localize("UTC", nonexistent="NaT", ambiguous="NaT")
    return s

def _ensure_sid_generated(df_gen: pd.DataFrame) -> pd.DataFrame:
    """Ensure generated dataframe has a track id column 'SID'."""
    df = df_gen.copy()
    if "SID" in df.columns:
        return df
    if "index" in df.columns:
        df["SID"] = df["index"].astype(str)
    else:
        # Fallback: give everything a single SID (least desirable)
        warnings.warn("No 'SID' or 'index' in generated data. Assigning a single SID.")
        df["SID"] = "GEN_0"
    return df

def _select_cols(df: pd.DataFrame, cols: Iterable[str]) -> pd.DataFrame:
    existing = [c for c in cols if c in df.columns]
    return df[existing].copy()

In [3]:
#Plot helpers

def _setup_map_ax(figsize=(7,4), extent=(-180, 180, -60, 60)):
    fig = plt.figure(figsize=figsize, constrained_layout=True)
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3)
    ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.5)
    return fig, ax

def _kde2d(x: np.ndarray, y: np.ndarray, xbins=180, ybins=120,
           xlim=(-180,180), ylim=(-60,60), bw_method=None) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    if len(x) < 10:
        return np.linspace(*xlim, xbins), np.linspace(*ylim, ybins), np.zeros((ybins, xbins))
    xx = np.linspace(*xlim, xbins)
    yy = np.linspace(*ylim, ybins)
    X, Y = np.meshgrid(xx, yy)
    kde = gaussian_kde(np.vstack([x, y]), bw_method=bw_method)
    Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(Y.shape)
    Z = Z / Z.sum()  # density normalized
    return xx, yy, Z

def _imshow_map(ax, xx, yy, Z, cmap="Oranges", label=None, vmin=None, vmax=None):
    extent = [xx.min(), xx.max(), yy.min(), yy.max()]
    img = ax.imshow(Z, extent=extent, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    if label is not None:
        cb = plt.colorbar(img, ax=ax, shrink=0.8)
        cb.set_label(label)
    return img

In [4]:
@dataclass
class IBTracks:
    df: pd.DataFrame

    @classmethod
    def from_df(cls, df: pd.DataFrame) -> "IBTracks":
        expected = ['SID','datetime','NAME','LAT','LON','BASIN',
                    'DIST2LAND','LANDFALL','wind','pres']
        missing = [c for c in expected if c not in df.columns]
        if missing:
            raise ValueError(f"IBTrACS missing columns: {missing}")

        dd = df.copy()
        dd["LAT"] = pd.to_numeric(dd["LAT"], errors="coerce")
        dd["LON"] = _wrap_lon(pd.to_numeric(dd["LON"], errors="coerce"))
        dd["wind"] = pd.to_numeric(dd["wind"], errors="coerce")  # kt or m/s? (use consistent unit across both)
        dd["pres"] = pd.to_numeric(dd["pres"], errors="coerce")
        dd["LANDFALL"] = dd["LANDFALL"].astype(str)

        # Genesis: first timestamp per SID
        dd.sort_values(["SID", "datetime"], inplace=True, kind="mergesort")
        genesis_idx = dd.groupby("SID", sort=False)["datetime"].idxmin()
        dd["is_genesis"] = False
        dd.loc[genesis_idx, "is_genesis"] = True

        # Landfall events per IB rule: LANDFALL == "0" marks "within next 6h"
        dd["is_landfall"] = dd["LANDFALL"].eq("0")

        # Lifetime in hours (per SID)
        life_hours = (dd.groupby("SID")["datetime"].transform("max") -
                      dd.groupby("SID")["datetime"].transform("min")).dt.total_seconds() / 3600.0
        dd["lifetime_hours"] = life_hours

        return cls(dd)

@dataclass
class GeneratedTracks:
    df: pd.DataFrame

    @classmethod
    def from_df(cls, df: pd.DataFrame) -> "GeneratedTracks":
        expected_some = ['lat_left','lon_left','datetime','basin','wind_speed','pc_hpa',
                         'is_on_land','dist2coast_meters','year','month']
        missing = [c for c in expected_some if c not in df.columns]
        if missing:
            warnings.warn(f"Generated tracks missing commonly used columns: {missing}. Proceeding with what’s available.")

        dd = df.copy()
        dd = _ensure_sid_generated(dd)
        # Time + coords
        if "datetime" in dd.columns:
            dd["datetime"] = _as_datetime(dd["datetime"])
        elif "time" in dd.columns:
            dd["datetime"] = _as_datetime(dd["time"])
        else:
            raise ValueError("Generated tracks need a datetime column ('datetime' or 'time').")

        lat = dd["lat_left"] if "lat_left" in dd.columns else dd.get("LAT", np.nan)
        lon = dd["lon_left"] if "lon_left" in dd.columns else dd.get("LON", np.nan)

        dd["LAT"] = pd.to_numeric(lat, errors="coerce")
        dd["LON"] = _wrap_lon(pd.to_numeric(lon, errors="coerce"))

        # Intensity
        if "wind_speed" in dd.columns:
            dd["wind"] = pd.to_numeric(dd["wind_speed"], errors="coerce")
        if "pc_hpa" in dd.columns:
            dd["pres"] = pd.to_numeric(dd["pc_hpa"], errors="coerce")
        elif "mslp_hpa" in dd.columns:
            dd["pres"] = pd.to_numeric(dd["mslp_hpa"], errors="coerce")

        # Basin naming consistency
        if "basin" in dd.columns:
            dd["BASIN"] = dd["basin"].astype(str)
        elif "BASIN" not in dd.columns:
            dd["BASIN"] = "UNKNOWN"

        # Sort & genesis
        dd.sort_values(["SID", "datetime"], inplace=True, kind="mergesort")
        genesis_idx = dd.groupby("SID", sort=False)["datetime"].idxmin()
        dd["is_genesis"] = False
        dd.loc[genesis_idx, "is_genesis"] = True

        # Landfall events: first time step that transitions offshore -> on_land
        if "is_on_land" in dd.columns:
            dd["is_on_land"] = dd["is_on_land"].astype(bool)
            dd["prev_on_land"] = dd.groupby("SID")["is_on_land"].shift(1).fillna(False)
            dd["is_landfall"] = (~dd["prev_on_land"]) & (dd["is_on_land"])
        else:
            # Fallback: use coastline distance
            if "dist2coast_meters" in dd.columns:
                d = pd.to_numeric(dd["dist2coast_meters"], errors="coerce")
                prev = d.groupby(dd["SID"]).shift(1)
                dd["is_landfall"] = (prev > 0) & (d <= 0)
            else:
                warnings.warn("No land indicator in generated data; landfall charts will be skipped.")
                dd["is_landfall"] = False

        # Lifetime in hours
        life_hours = (dd.groupby("SID")["datetime"].transform("max") -
                      dd.groupby("SID")["datetime"].transform("min")).dt.total_seconds() / 3600.0
        dd["lifetime_hours"] = life_hours

        return cls(dd)

In [5]:
def genesis_density_hist2d(
    ib: IBTracks,
    gen: GeneratedTracks,
    savepath: Optional[str] = None,
    extent: tuple[float, float, float, float] = (-180, 180, -60, 60),
    res_deg: float = 2.0,           # grid resolution in degrees (lon/lat)
    density: bool = True,           # match KDE-like density; set False for raw counts
):
    """
    2D-histogram comparison of genesis locations (observed vs generated).

    Parameters
    ----------
    res_deg : float
        Bin size in degrees (same for lon & lat). Smaller -> finer grid.
    density : bool
        If True, histogram is normalized to a probability density (like KDE).
        If False, shows raw counts per bin.
    """
    # --- Extract genesis lon/lat
    ib_g  = ib.df.loc[ib.df["is_genesis"], ["LON", "LAT"]].dropna()
    gen_g = gen.df.loc[gen.df["is_genesis"], ["LON", "LAT"]].dropna()

    # --- Build bin edges from extent and resolution
    xmin, xmax, ymin, ymax = extent
    nx = max(1, int(np.ceil((xmax - xmin) / res_deg)))
    ny = max(1, int(np.ceil((ymax - ymin) / res_deg)))

    xedges = np.linspace(xmin, xmax, nx + 1)
    yedges = np.linspace(ymin, ymax, ny + 1)

    # --- 2D histogram (note: np.histogram2d expects x first, y second)
    H_ib,  xE_ib,  yE_ib  = np.histogram2d(ib_g["LON"].values,  ib_g["LAT"].values,
                                           bins=[xedges, yedges], density=density)
    H_ge,  xE_ge,  yE_ge  = np.histogram2d(gen_g["LON"].values, gen_g["LAT"].values,
                                           bins=[xedges, yedges], density=density)

    # --- Convert edges -> bin centers (for _imshow_map compatibility)
    xcenters = 0.5 * (xedges[:-1] + xedges[1:])
    ycenters = 0.5 * (yedges[:-1] + yedges[1:])
    xx, yy = np.meshgrid(xcenters, ycenters)  # yy rows correspond to LAT

    # Align orientation for plotting: transpose so Z[i,j] ~ yy[i], xx[j]
    Z_ib = H_ib.T
    Z_ge = H_ge.T

    # --- Common color scale
    vmax = max(Z_ib.max() if Z_ib.size else 0.0,
               Z_ge.max() if Z_ge.size else 0.0)

    # --- Figure & maps
    fig = plt.figure(figsize=(12, 4), constrained_layout=True)
    proj = ccrs.PlateCarree()

    for i, (Z, title) in enumerate(
        [(Z_ib, "IBTrACS Genesis (2D histogram)"),
         (Z_ge, "Generated Genesis (2D histogram)")],
        start=1
    ):
        ax = fig.add_subplot(1, 2, i, projection=proj)
        ax.set_extent(extent, crs=proj)

        # Land/ocean base
        ax.add_feature(cfeature.LAND, facecolor="0.95", zorder=0)
        ax.add_feature(cfeature.OCEAN, facecolor="white", zorder=0)

        # Coastlines + borders
        ax.add_feature(cfeature.COASTLINE, linewidth=0.6, zorder=3)
        ax.add_feature(cfeature.BORDERS, linewidth=0.5, zorder=3)

        countries = NaturalEarthFeature(
            category="cultural", name="admin_0_countries", scale="50m"
        )
        ax.add_feature(countries, edgecolor="black", facecolor="none",
                       linewidth=0.4, zorder=3)

        # Density image (uses your existing helper)
        _imshow_map(ax, xx, yy, Z, label=("Density" if density else "Count"), vmax=vmax)

        ax.set_title(title)
        gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5)
        try:
            gl.top_labels = gl.right_labels = False
        except Exception:
            pass

    if savepath:
        fig.savefig(savepath, dpi=200)

    return fig


In [6]:
def landfall_histograms(ib: IBTracks, gen: GeneratedTracks,
                        bin_edges_lon=np.linspace(-180,180,37),
                        savepath: Optional[str] = None):
    """Histogram of landfall longitudes (proxy for location) comparing IB vs Gen."""
    ib_lf = ib.df.loc[ib.df["is_landfall"], ["LON"]].dropna()
    ge_lf = gen.df.loc[gen.df["is_landfall"], ["LON"]].dropna()

    fig, ax = plt.subplots(figsize=(8,4), constrained_layout=True)
    ax.hist(ib_lf["LON"].values, bins=bin_edges_lon, alpha=0.6, label="IBTrACS", density=True)
    ax.hist(ge_lf["LON"].values, bins=bin_edges_lon, alpha=0.6, label="Generated", density=True)
    ax.set_xlabel("Landfall Longitude"); ax.set_ylabel("Density")
    ax.set_title("Landfall Location Distribution")
    ax.legend()
    if savepath:
        fig.savefig(savepath, dpi=200)
    return fig

def intensity_distributions(ib: IBTracks, gen: GeneratedTracks,
                            savepath: Optional[str] = None):
    """PDF/CDF of max wind and min pressure per storm."""
    def _storm_extremes(df):
        grp = df.groupby("SID", sort=False)
        max_wind = grp["wind"].max()
        min_pres = grp["pres"].min()
        return max_wind, min_pres

    ib_w, ib_p = _storm_extremes(ib.df)
    ge_w, ge_p = _storm_extremes(gen.df)

    fig, axs = plt.subplots(2, 2, figsize=(10,6), constrained_layout=True)
    # PDFs
    axs[0,0].hist(ib_w.dropna(), bins=30, density=True, alpha=0.6, label="IBTrACS")
    axs[0,0].hist(ge_w.dropna(), bins=30, density=True, alpha=0.6, label="Generated")
    axs[0,0].set_title("Max Wind PDF"); axs[0,0].set_xlabel("Wind"); axs[0,0].set_ylabel("Density"); axs[0,0].legend()

    axs[0,1].hist(ib_p.dropna(), bins=30, density=True, alpha=0.6, label="IBTrACS")
    axs[0,1].hist(ge_p.dropna(), bins=30, density=True, alpha=0.6, label="Generated")
    axs[0,1].set_title("Min Pressure PDF"); axs[0,1].set_xlabel("Pressure (hPa)"); axs[0,1].set_ylabel("Density"); axs[0,1].legend()
    # CDFs
    for ax, series, title in [(axs[1,0], ib_w.dropna().sort_values(), "Max Wind CDF (IBTrACS)"),
                              (axs[1,0], ge_w.dropna().sort_values(), "Max Wind CDF (Generated)")]:
        y = np.linspace(0,1,len(series), endpoint=False)
        ax.plot(series.values, y, label=title.split(" (")[1][:-1])
    axs[1,0].set_title("Max Wind CDF"); axs[1,0].set_xlabel("Wind"); axs[1,0].set_ylabel("F"); axs[1,0].legend()

    for ax, series, title in [(axs[1,1], ib_p.dropna().sort_values(), "Min Pres CDF (IBTrACS)"),
                              (axs[1,1], ge_p.dropna().sort_values(), "Min Pres CDF (Generated)")]:
        y = np.linspace(0,1,len(series), endpoint=False)
        ax.plot(series.values, y, label=title.split(" (")[1][:-1])
    axs[1,1].set_title("Min Pressure CDF"); axs[1,1].set_xlabel("Pressure (hPa)"); axs[1,1].set_ylabel("F"); axs[1,1].legend()

    if savepath:
        fig.savefig(savepath, dpi=200)
    return fig

def lifetime_distribution(ib: IBTracks, gen: GeneratedTracks,
                          savepath: Optional[str] = None):
    """Storm lifetime (hours -> days) distribution."""
    ib_days = (ib.df.groupby("SID")["lifetime_hours"].max()/24.0).dropna()
    ge_days = (gen.df.groupby("SID")["lifetime_hours"].max()/24.0).dropna()

    fig, ax = plt.subplots(figsize=(8,4), constrained_layout=True)
    ax.hist(ib_days, bins=30, density=True, alpha=0.6, label="IBTrACS")
    ax.hist(ge_days, bins=30, density=True, alpha=0.6, label="Generated")
    ax.set_xlabel("Lifetime (days)"); ax.set_ylabel("Density")
    ax.set_title("Storm Lifetime Distribution")
    ax.legend()
    if savepath:
        fig.savefig(savepath, dpi=200)
    return fig

def seasonal_counts_by_basin(
    ib: IBTracks,
    gen: GeneratedTracks,
    seed_col: str | None = "seed",
    basin_col: str = "BASIN",
    band: str = "quantile",           # "std" or "quantile"
    q_low: float = 0.1, q_high: float = 0.9,
    basins: list[str] | None = None,  # filter/order specific basins; default = all found
    ncols: int = 3,
    figsize_per_panel: tuple[float, float] = (4.0, 3.0),
    sharex: bool = True,
    sharey: bool = True,
    savepath: Optional[str] = None,
):
    """
    Seasonal (calendar-year) storm counts by basin.
      - One subplot per basin (faceted).
      - IBTrACS shown as bars.
      - Generated:
          * If `seed_col` present -> mean curve across seeds + uncertainty band (std or [q_low, q_high]).
          * Else -> single bar series (no seeds).
    """
    # ---- Helper: year per row
    def _year_series(df):
        if "year" in df.columns:
            return pd.to_numeric(df["year"], errors="coerce")
        if "datetime" in df.columns:
            return pd.to_datetime(df["datetime"]).dt.year
        raise KeyError("Expect a 'year' column or 'datetime' column in the input dataframes.")

    # Make working copies
    ib_df = ib.df.copy()
    g_df  = gen.df.copy()

    # Ensure required columns exist
    for name, df in [("IBTrACS", ib_df), ("Generated", g_df)]:
        if basin_col not in df.columns:
            raise KeyError(f"{name} dataframe is missing column '{basin_col}'.")
        if "SID" not in df.columns:
            raise KeyError(f"{name} dataframe is missing column 'SID'.")

    # Year columns
    ib_year = _year_series(ib_df)
    g_year  = _year_series(g_df)

    # De-duplicate to 1 row per storm (SID) and assign year/basin
    ib_unique = (ib_df.drop_duplicates(["SID"])
                       .assign(year=ib_year)
                       .dropna(subset=["year"]))
    g_unique  = (g_df.drop_duplicates(["SID"])
                      .assign(year=g_year)
                      .dropna(subset=["year"]))

    # Determine basins to plot
    if basins is None:
        basins = sorted(pd.Index(ib_unique[basin_col].unique()).union(g_unique[basin_col].unique()).dropna().tolist())

    # Figure layout
    nb = len(basins)
    nrows = math.ceil(nb / ncols) if nb else 1
    fig_w = max(ncols * figsize_per_panel[0], 4.0)
    fig_h = max(nrows * figsize_per_panel[1], 3.0)
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), sharex=sharex, sharey=sharey, constrained_layout=True)
    if nb == 1:
        axes = np.array([axes])  # make iterable
    axes = axes.flatten()

    # Common x-axis ticks based on union of years across both datasets (per basin we’ll reindex)
    all_years = pd.Index(sorted(pd.concat([ib_unique["year"], g_unique["year"]]).unique()))

    # Plot each basin
    for ax, basin in zip(axes, basins):
        ib_b = ib_unique.loc[ib_unique[basin_col] == basin]
        g_b  = g_unique.loc[g_unique[basin_col] == basin]

        # --- IBTrACS counts (single reality)
        ib_counts = (ib_b.groupby("year").size().rename("count").reindex(all_years, fill_value=0))

        # --- Generated
        if seed_col and (seed_col in g_b.columns):
            # counts per (seed, year)
            per_seed = (
                g_b.groupby([seed_col, "year"]).size()
                   .rename("count")
                   .reset_index()
            )
            # pivot: rows=year, cols=seed
            pivot = (per_seed.pivot(index="year", columns=seed_col, values="count")
                             .reindex(all_years))
            years = pivot.index.to_numpy()

            mean_counts = pivot.mean(axis=1, skipna=True)
            if band == "std":
                std_counts = pivot.std(axis=1, ddof=1)
                lo = (mean_counts - std_counts).clip(lower=0)
                hi = (mean_counts + std_counts)
                band_label = "Seed variability (±1σ)"
            else:
                lo = pivot.quantile(q_low, axis=1)
                hi = pivot.quantile(q_high, axis=1)
                band_label = f"Seed variability [{q_low:.2f}, {q_high:.2f}]"

            # Plot: IB bars, Generated mean + band
            ax.bar(years - 0.2, ib_counts.values, width=0.4, label="IBTrACS")
            ax.plot(years, mean_counts.values, label="Generated (mean across seeds)")
            ax.fill_between(years, lo.values, hi.values, alpha=0.25, label=band_label)
        else:
            # No seeds -> just bar counts per year
            gen_counts = (g_b.groupby("year").size().rename("count").reindex(all_years, fill_value=0))
            years = gen_counts.index.to_numpy()

            # Side-by-side bars
            ax.bar(years - 0.15, ib_counts.values, width=0.3, label="IBTrACS")
            ax.bar(years + 0.15, gen_counts.values, width=0.3, label="Generated")

        ax.set_title(f"Basin: {basin}")
        ax.set_xlabel("Year")
        ax.set_ylabel("# Storms")
        ax.grid(True, axis="y", alpha=0.25)

    # Hide any unused axes
    for j in range(len(basins), len(axes)):
        fig.delaxes(axes[j])

    # One legend for all (outside layout-friendly)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=min(4, len(labels)))

    fig.suptitle("Seasonal Storm Counts by Basin (seed-aware)", y=1.02)

    if savepath:
        fig.savefig(savepath, dpi=200, bbox_inches="tight")

    return fig

from scipy.stats import genextreme

def extreme_value_return_levels(ib: IBTracks, gen: GeneratedTracks,
                                variable: str = "wind",       # 'wind' (max) or 'pres' (min)
                                block: str = "year",
                                seed_col: str | None = "seed",
                                T: Iterable[float] = (2,5,10,20,50,100,200),
                                band: str = "quantile",       # "std" or "quantile"
                                q_low: float = 0.1, q_high: float = 0.9,
                                min_years_per_seed: int = 8,
                                savepath: Optional[str] = None):
    """
    Seed-aware return level plot via block maxima + GEV fit.
    - IBTrACS: single curve (one reality).
    - Generated: fit GEV per seed using that seed's annual maxima, then
      aggregate return levels across seeds: mean + band (std or quantiles).
    """
    T = np.asarray(list(T), dtype=float)
    assert variable in ("wind", "pres"), "variable must be 'wind' or 'pres'"

    def _block_max_per_year(df, var):
        # per-storm max, then map storm to its first year, then max over storms per year
        if var == "pres":
            ser = -pd.to_numeric(df["pres"], errors="coerce")  # maxima of -pres = minima of pres
        else:
            ser = pd.to_numeric(df["wind"], errors="coerce")
        years = (pd.to_numeric(df["year"], errors="coerce")
                 if "year" in df.columns else df["datetime"].dt.year)
        per_storm = df.assign(val=ser).groupby("SID", sort=False)["val"].max()
        storm_year = df.groupby("SID", sort=False)["datetime"].min().dt.year if "datetime" in df.columns else years.groupby(df["SID"]).min()
        bm = per_storm.groupby(storm_year).max().dropna().sort_index()
        return bm

    def _fit_gev(data):
        # Returns (c, loc, scale) or None if insufficient
        try:
            if len(data) < 3:
                return None
            return genextreme.fit(np.asarray(data, dtype=float))
        except Exception:
            return None

    def _rl_from_fit(T, fit_params):
        if fit_params is None:
            return np.full_like(T, np.nan, dtype=float)
        c, loc, scale = fit_params
        q = 1.0 - 1.0/T
        return genextreme.ppf(q, c, loc=loc, scale=scale)

    # ---- IBTrACS single reality
    ib_bm = _block_max_per_year(ib.df, variable)
    ib_fit = _fit_gev(ib_bm.values)
    ib_rl = _rl_from_fit(T, ib_fit)

    # ---- Generated (seed-aware)
    g = gen.df.copy()
    if seed_col and seed_col in g.columns:
        rl_per_seed = []
        for s, g_s in g.groupby(seed_col, sort=False):
            bm_s = _block_max_per_year(g_s, variable)
            if bm_s.index.nunique() < min_years_per_seed:
                continue
            fit_s = _fit_gev(bm_s.values)
            rl_s = _rl_from_fit(T, fit_s)
            rl_per_seed.append(rl_s)

        rl_per_seed = np.asarray(rl_per_seed, dtype=float)
        # Aggregate across seeds
        mean_rl = np.nanmean(rl_per_seed, axis=0) if rl_per_seed.size else np.full_like(T, np.nan)
        if band == "std" and rl_per_seed.size:
            std_rl = np.nanstd(rl_per_seed, axis=0, ddof=1)
            lo, hi = mean_rl - std_rl, mean_rl + std_rl
        else:
            if rl_per_seed.size:
                lo = np.nanquantile(rl_per_seed, q_low, axis=0)
                hi = np.nanquantile(rl_per_seed, q_high, axis=0)
            else:
                lo = hi = np.full_like(T, np.nan)
    else:
        # No seeds provided: fit once on pooled data (legacy behavior)
        bm = _block_max_per_year(g, variable)
        fit = _fit_gev(bm.values)
        mean_rl = _rl_from_fit(T, fit)
        lo = hi = mean_rl  # no uncertainty

    # ---- Plot
    fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
    if np.all(np.isfinite(ib_rl)):
        ax.plot(T, ib_rl, marker="o", label="IBTrACS")
    else:
        ax.plot([], [], label="IBTrACS (insufficient data)")

    if seed_col and (np.any(np.isfinite(mean_rl))):
        ax.plot(T, mean_rl, marker="o", label="Generated (mean across seeds)")
        if not np.allclose(lo, hi, equal_nan=True):
            ax.fill_between(T, lo, hi, alpha=0.25, label=f"Seed variability ({band})")
    else:
        ax.plot(T, mean_rl, marker="o", label="Generated")

    ax.set_xscale("log")
    ax.set_xlabel("Return period (years)")
    ylab = "Return level" if variable == "wind" else "Return level (on -pres scale)"
    ax.set_ylabel(ylab)
    ttl_var = "Max Wind" if variable == "wind" else "Min Pressure (as -pres)"
    ax.set_title(f"Extreme Value Return Levels — {ttl_var} (seed-aware)")
    ax.grid(True, which="both", alpha=0.3, linewidth=0.5)
    ax.legend()
    if savepath:
        fig.savefig(savepath, dpi=200)
    return fig, {"T": T, "ib_fit": ib_fit}


def ecdf_by_basin(ib: IBTracks,
                  gen: GeneratedTracks,
                  seed_col: str | None = "seed",
                  basin_col: str = "BASIN",
                  variable: str = "wind",            # only 'wind' supported here
                  mode: str = "per_storm_max",       # 'per_storm_max' or 'all_steps'
                  q_low: float = 0.1,
                  q_high: float = 0.9,
                  n_grid: int = 400,
                  basins: Optional[Iterable[str]] = None,
                  savepath: Optional[str] = None):
    """
    ECDF comparison of wind speed by basin.
    - IBTrACS: single ECDF per basin (one reality).
    - Generated: mean ECDF across seeds with confidence band (quantile [q_low, q_high]).
    - mode:
        * 'per_storm_max' -> ECDF of per-storm maximum wind (recommended)
        * 'all_steps'     -> ECDF over all time steps
    """
    assert variable == "wind", "Only 'wind' is supported in this ECDF function."

    def _series_by_mode(df, basin_name=None, threshold=20.0):
        d = df if basin_name is None else df[df[basin_col] == basin_name]
        w = pd.to_numeric(d["wind"], errors="coerce")

        if mode == "per_storm_max":
            return d.assign(w=w).groupby("SID", sort=False)["w"].max().dropna().values
        elif mode == "all_steps":
            return w[(w.notna()) & (w > threshold)].values  # strictly above 20
        else:
            raise ValueError("mode must be 'per_storm_max' or 'all_steps'")

    def _ecdf_on_grid(sample, grid):
        # returns ECDF(sample) evaluated at each grid point
        if sample is None or len(sample) == 0:
            return np.full_like(grid, np.nan, dtype=float)
        s = np.sort(np.asarray(sample, dtype=float))
        return np.searchsorted(s, grid, side="right") / s.size
    
    def _on_land_ib(df):
        # coerce LANDFALL to numeric and apply < 10 filter
        landfall = pd.to_numeric(df.get("LANDFALL"), errors="coerce")
        return df[landfall < 10]

    def _on_land_gen(df):
        # coerce is_in_land to boolean; treat NaN as False
        land = df.get("is_in_land")
        if land is None:
            # fallback: if your column is named differently, adjust here
            land = df.get("is_on_land")
        land_bool = pd.Series(land, index=df.index).astype("boolean").fillna(False)
        return df[land_bool]

    # Determine basins to plot
    #ib_on_land = _on_land_ib(ib.df).copy()
    #gen_on_land= _on_land_gen(gen.df).copy()

    ib_on_land = ib.df.copy()
    gen_on_land = gen.df.copy()

    ib_basins = ib_on_land[basin_col].dropna().unique().tolist()
    ge_basins = gen_on_land[basin_col].dropna().unique().tolist()
    if basins is None:
        basins = sorted(set(ib_basins).union(ge_basins))

    # Build global x-grid for wind to ensure curves align
    # Use both IB and generated winds to set range (robust to tails)
    ib_all_w = pd.to_numeric(ib_on_land["wind"], errors="coerce")
    ge_all_w = pd.to_numeric(gen_on_land["wind"], errors="coerce")
    all_w = pd.concat([ib_all_w, ge_all_w], axis=0).dropna()
    if all_w.empty:
        raise ValueError("No wind data available to build ECDF grid.")
    x_min, x_max = np.nanpercentile(all_w, [0.5, 99.5])  # trim extreme outliers for axis
    if not np.isfinite(x_min) or not np.isfinite(x_max) or x_min >= x_max:
        x_min, x_max = float(np.nanmin(all_w)), float(np.nanmax(all_w))
    grid = np.linspace(x_min, x_max, n_grid)

    # Layout subplots
    n = len(basins)
    ncols = min(3, max(1, int(np.ceil(np.sqrt(n)))))
    nrows = int(np.ceil(n / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows), constrained_layout=True)
    axs = np.atleast_1d(axs).ravel()

    for i, basin in enumerate(basins):
        ax = axs[i]

        # IBTrACS: single ECDF
        ib_sample = _series_by_mode(ib_on_land, basin)
        ib_ecdf = _ecdf_on_grid(ib_sample, grid)
        if np.any(np.isfinite(ib_ecdf)):
            ax.step(grid, ib_ecdf, where="post", label="IBTrACS", linewidth=1.5)

        # Generated: seed-aware ECDF band
        g = gen_on_land
        g = g[g[basin_col] == basin]
        if seed_col and seed_col in g.columns:
            ecdf_seed = []
            for s, g_s in g.groupby(seed_col, sort=False):
                sample_s = _series_by_mode(g_s, None)
                ecdf_s = _ecdf_on_grid(sample_s, grid)
                if np.any(np.isfinite(ecdf_s)):
                    ecdf_seed.append(ecdf_s)

            if len(ecdf_seed) > 0:
                M = np.vstack(ecdf_seed)
                mean_ecdf = np.nanmean(M, axis=0)
                lo = np.nanquantile(M, q_low, axis=0)
                hi = np.nanquantile(M, q_high, axis=0)

                ax.step(grid, mean_ecdf, where="post", label="Generated (mean over seeds)")
                ax.fill_between(grid, lo, hi, step="post", alpha=0.25,
                                label=f"Seed band [{int(q_low*100)}–{int(q_high*100)}%]")
        else:
            # No seed column: single ECDF on pooled generated data
            ge_sample = _series_by_mode(g, None)
            ge_ecdf = _ecdf_on_grid(ge_sample, grid)
            if np.any(np.isfinite(ge_ecdf)):
                ax.step(grid, ge_ecdf, where="post", label="Generated", linewidth=1.5)

        ax.set_title(f"{basin} — ECDF of {'Max Wind per Storm' if mode=='per_storm_max' else 'All-Step Wind'}")
        ax.set_xlabel("Wind speed")
        ax.set_ylabel("ECDF")
        ax.set_xlim(grid[0], grid[-1])
        ax.set_ylim(0, 1)
        ax.grid(True, alpha=0.3, linewidth=0.5)
        ax.legend()

    # Hide any unused axes
    for j in range(i+1, len(axs)):
        axs[j].set_visible(False)

    if savepath:
        fig.savefig(savepath, dpi=200)
    return fig


from matplotlib.collections import LineCollection
import matplotlib.colors as mcolors

def plot_track_intensity_map(
    ib: IBTracks | None = None,
    gen: GeneratedTracks | None = None,
    which: str = "generated",          # "generated" or "ib"
    basin: str | None = None,          # filter one basin, e.g. "North Atlantic"
    seed_col: str | None = "seed",
    seeds: list | None = None,         # e.g. [0,1,2]; None = all
    extent: tuple = (-180, 180, -60, 60),
    vmin: float | None = None,         # wind color scale
    vmax: float | None = None,
    cmap: str = "viridis",
    alpha: float = 0.9,
    lw: float = 0.6,
    max_tracks: int | None = None,     # subsample tracks for speed, e.g. 4000
    title: str | None = None,
    savepath: str | None = None,
):
    """
    Plot cyclone tracks with color = wind speed.
    - which: choose "generated" or "ib".
    - basin: optional basin filter by column BASIN.
    - seeds: optional list of seeds for generated data.
    - extent: (lon_min, lon_max, lat_min, lat_max).
    """
    assert which in ("generated", "ib")
    df = gen.df.copy() if which == "generated" else ib.df.copy()

    # Filters
    if basin is not None and "BASIN" in df.columns:
        df = df[df["BASIN"] == basin]
    if which == "generated" and seed_col and seeds is not None and seed_col in df.columns:
        df = df[df[seed_col].isin(seeds)]

    # Clean / prep
    df = df[["SID","datetime","LON","LAT","wind"]].dropna(subset=["SID","datetime","LON","LAT","wind"]).copy()
    df["wind"] = pd.to_numeric(df["wind"], errors="coerce")
    df = df.dropna(subset=["wind"])
    # Sort within tracks
    df = df.sort_values(["SID","datetime"], kind="mergesort")

    # Optional subsample by tracks
    if max_tracks is not None:
        keep_sids = df["SID"].drop_duplicates().sample(min(max_tracks, df["SID"].nunique()), random_state=0)
        df = df[df["SID"].isin(keep_sids)]

    # Build line segments (Nx2x2) and colors from segment-mean wind
    segs = []
    cols = []
    for sid, g in df.groupby("SID", sort=False):
        xy = g[["LON","LAT"]].to_numpy()
        if len(xy) < 2:
            continue
        w = g["wind"].to_numpy()
        segs_sid = np.stack([xy[:-1], xy[1:]], axis=1)  # (n-1, 2, 2)
        w_mid = 0.5*(w[:-1] + w[1:])
        segs.append(segs_sid)
        cols.append(w_mid)
    if not segs:
        raise ValueError("No segments to plot after filtering.")
    segs = np.vstack(segs)
    cols = np.concatenate(cols)

    # Color norm
    if vmin is None: vmin = np.nanpercentile(cols, 1)
    if vmax is None: vmax = np.nanpercentile(cols, 99)
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

    # Make figure / axes (with Cartopy if available)

    fig = plt.figure(figsize=(12,6), constrained_layout=True)
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor="0.95", zorder=0)
    ax.add_feature(cfeature.OCEAN, facecolor="white", zorder=0)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6, zorder=3)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4, zorder=3)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5)
    try:
        gl.top_labels = gl.right_labels = False
    except Exception:
        pass


    lc = LineCollection(segs, cmap=cmap, norm=norm, alpha=alpha, linewidths=lw)
    lc.set_array(cols)
    ax.add_collection(lc)
    cb = plt.colorbar(lc, ax=ax, shrink=0.85, pad=0.02)
    cb.set_label("Wind speed")

    ttl = title or (
        f"{'Generated' if which=='generated' else 'IBTrACS'} tracks — color: wind"
        + (f" — {basin}" if basin else "")
    )
    ax.set_title(ttl)

    if savepath:
        fig.savefig(savepath, dpi=200)
    return fig


In [8]:
catherina_fit_path = Path(
    "../../data/input/fit/Catherina_fit.db"
)

method = "RMSF"
config_path =  "../config.toml"

with open(
    config_path,
    "rb",
) as f:  # Open the file in binary mode
    config_files = tomllib.load(f)

#Load data
#Read and preprocess ibtracks
main_config = config_files["main_params"]
gen_config = config_files["generation"]
data_dir = ".." / Path(main_config["input_data_dir"])
ibtracksf_folder = data_dir / gen_config["ibtracs_path"]

#New version
ibtracs = read_ibtracs(fpath=ibtracksf_folder, signed_coords=True)
ibtracs = preprocess_ibtracs(ibtracs, 32.92)   

# Load tracks data

tracks_historical_dir = Path("../../data/input/catherina_historical/intensified_tracks/ACCESS-CM2/historical/")
dataset = pds.dataset(tracks_historical_dir, 
                      format="parquet",
                        partitioning="hive")
part_fields = {f.name: f.type for f in dataset.partitioning.schema}

#year_f = pc.cast(pc.field("year"), pa.int32()) if part_fields.get("year") == pa.string() else pc.field("year")
#month_f = pc.cast(pc.field("month"), pa.int32()) if part_fields.get("month") == pa.string() else pc.field("month")
seed_f = pc.cast(pc.field("seed"), pa.int64())  if part_fields.get("seed") == pa.string() else pc.field("seed")

filt = seed_f.isin(list(range(20)))

#Avoid validity checks in the Scanner if columns may be absent; filter later
scanner = pds.Scanner.from_dataset(dataset, filter=filt)

tracks_historical = scanner.to_table().to_pandas()

tracks_historical = tracks_historical.sort_values(["SID", "datetime"], kind="mergesort").copy().reset_index()
# #tracks_historical = tracks_historical.reset_index().drop_duplicates(['SID','step'])

# to_drop = tracks_historical.groupby("SID")['SST'].transform(lambda s: s.le(15).cummax())
# tracks_historical_filtered  = tracks_historical[~to_drop]
#Condition on max wind
max_wind_df = tracks_historical.groupby("SID")["final_wind_speed"].max()

# filtering out storms with wind speed inferior to threshold
tracks_to_keep = max_wind_df[(max_wind_df >= 32.92)].index
tracks_historical_filtered = tracks_historical.loc[tracks_historical["SID"].isin(tracks_to_keep)]




2025-12-19 16:39:08.842 | INFO     | src.genesis.genesis_utils:input_processing:464 - initiating basic input processing...
2025-12-19 16:39:09.545 | INFO     | src.genesis.genesis_utils:get_average_pres:488 - retrieving the pressure from IBTrACS...
2025-12-19 16:39:09.766 | INFO     | src.genesis.genesis_utils:get_average_wind:496 - retrieving and convert winds from IBTrACS...
2025-12-19 16:39:10.321 | INFO     | src.genesis.genesis_utils:filter_wind:513 - filtering storm with wind above threshold...


FileNotFoundError: ../../data/input/catherina_historical/intensified_tracks/ACCESS-CM2/historical

In [9]:
ib = IBTracks.from_df(ibtracs)
ge = GeneratedTracks.from_df(tracks_historical_filtered)
map_extent = (-180,180,-60,60)

NameError: name 'tracks_historical_filtered' is not defined

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def make_genesis_probability_map(
    df: pd.DataFrame,
    *,
    grid_deg: float = 2.0,              # grid size in degrees (e.g., 1.0 or 2.0)
    lat_range: tuple[float, float] = (-60, 60),
    lon_range: tuple[float, float] = (-180, 180),
    basin: str | None = None,           # e.g. "ATL", "EP", "WP", "NI", "SI", "SP"
    years: tuple[int, int] | None = None,
    months: set[int] | list[int] | None = None,
    title: str | None = "Cyclone Genesis Probability",
):
    """
    Plot probability of cyclone genesis on a lat-lon grid from point data.
    Expects df with columns: ['lat','lon', 'basin', 'year', 'month', ...].
    Each row is one genesis point (one genesis per SID/seed/step).

    Returns (fig, ax).
    """
    # -------- filter data (optional) --------
    data = df.copy()
    if basin is not None and "basin" in data.columns:
        data = data[data["basin"] == basin]
    if years is not None and {"year"}.issubset(data.columns):
        y0, y1 = years
        data = data[(data["year"] >= y0) & (data["year"] <= y1)]
    if months is not None and {"month"}.issubset(data.columns):
        months = set(months)
        data = data[data["month"].isin(months)]

    # Guard
    if data.empty:
        raise ValueError("No genesis points after filtering; nothing to plot.")

    # Ensure numeric lat/lon and drop NaNs
    data = data[pd.notna(data["lat"]) & pd.notna(data["lon"])].copy()
    lat = data["lat"].astype(float).to_numpy()
    lon = data["lon"].astype(float).to_numpy()

    # Wrap longitudes to [-180, 180]
    lon = ((lon + 180) % 360) - 180

    # -------- build grid & histogram --------
    lat_min, lat_max = lat_range
    lon_min, lon_max = lon_range
    lat_edges = np.arange(lat_min, lat_max + grid_deg, grid_deg, dtype=float)
    lon_edges = np.arange(lon_min, lon_max + grid_deg, grid_deg, dtype=float)

    # counts per cell
    H, lon_edges_out, lat_edges_out = np.histogram2d(
        lon, lat, bins=[lon_edges, lat_edges]
    )
    # Convert counts to probability (per cell)
    total = H.sum()
    if total <= 0:
        raise ValueError("Histogram is empty; check filters / ranges.")
    P = H / total  # probability mass per cell

    # For pcolormesh, build cell centers (optional; edges are fine)
    lon_centers = 0.5 * (lon_edges_out[:-1] + lon_edges_out[1:])
    lat_centers = 0.5 * (lat_edges_out[:-1] + lat_edges_out[1:])

    # -------- plot --------
    # Try Cartopy for coastlines; fall back to plain Matplotlib
    try:
        import cartopy.crs as ccrs
        import cartopy.feature as cfeature
        proj = ccrs.PlateCarree()
        fig = plt.figure(figsize=(10, 5))
        ax = plt.axes(projection=proj)
        # pcolormesh expects edges in PlateCarree
        mesh = ax.pcolormesh(lon_edges_out, lat_edges_out, P.T, transform=proj)
        ax.coastlines(linewidth=0.8)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4)
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=proj)
        cb = plt.colorbar(mesh, ax=ax, shrink=0.8)
        cb.set_label("Genesis probability per grid cell")
        ax.set_title(title or "Cyclone Genesis Probability")
        # gridlines (optional)
        gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
        gl.right_labels = False
        gl.top_labels = False
    except Exception:
        # Plain Matplotlib fallback (no map projection)
        fig, ax = plt.subplots(figsize=(10, 5))
        extent = [lon_edges_out.min(), lon_edges_out.max(), lat_edges_out.min(), lat_edges_out.max()]
        im = ax.imshow(
            P.T,
            origin="lower",
            extent=extent,
            aspect="auto",
        )
        cb = plt.colorbar(im, ax=ax, shrink=0.8)
        cb.set_label("Genesis probability per grid cell")
        ax.set_xlim(lon_min, lon_max)
        ax.set_ylim(lat_min, lat_max)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        ax.set_title(title or "Cyclone Genesis Probability")
        ax.grid(alpha=0.3)

    return fig, ax

In [ ]:
genesis_ds = pds.dataset(
    Path("../../data/input/catherina_historical/genesis/"),
    format="parquet",
    partitioning="hive",
)
df = genesis_ds.to_table().to_pandas()
make_genesis_probability_map(df)

In [ ]:
make_genesis_probability_map(tracks_historical.drop_duplicates('SID', keep='first').rename(columns={'lat_left':'lat', 'lon_left':'lon'}))

In [ ]:
plot1 = genesis_density_hist2d(ib, ge, extent=map_extent)

In [ ]:
plot_landfall = landfall_histograms(ib, ge)

In [ ]:
plot_intensity = intensity_distributions(ib, ge)

In [ ]:
plot_lifetime = lifetime_distribution(ib, ge)

In [ ]:
plot_seasons = seasonal_counts_by_basin(ib, ge)

In [ ]:
fig_eva_w, fits_w = extreme_value_return_levels(ib, ge, variable="wind")

In [ ]:
fig = ecdf_by_basin(ib, ge,
                    seed_col="seed",
                    mode="per_storm_max",
                    q_low=0.1, q_high=0.9)

In [ ]:
track_intensity_plot = plot_track_intensity_map(ib, ge, which="generated",seed_col="seed", seeds=[0],
                         extent=(-180,180,-60,60))

In [ ]:
track_intensity_plot = plot_track_intensity_map(ib, ge, which="ib",
                         extent=(-180,180,-60,60))

In [ ]:
# Build dataset
dataset = pds.dataset(Path("../../data/input/catherina_ssp585/intensified_tracks_test/ACCESS-CM2/ssp585/"), format="parquet", partitioning="hive")

# Partition field types may be strings; cast to numeric for comparisons
part_fields = {f.name: f.type for f in dataset.partitioning.schema}

def _cast_part(name, want_type):
    f = pc.field(name)
    return pc.cast(f, want_type) if part_fields.get(name) == pa.string() else f

seed_f = _cast_part("seed", pa.int64())
year_f = _cast_part("year", pa.int32())

# Base filter: seed < 100
base = seed_f < pa.scalar(20, pa.int64())

# Year slices
filt_2025_2050 = base & (year_f < pa.scalar(2050, pa.int32()))
filt_2050_2075 = base & (year_f >= pa.scalar(2050, pa.int32())) & (year_f < pa.scalar(2075, pa.int32()))
filt_2075_2100 = base & (year_f >= pa.scalar(2075, pa.int32()))

# Only pull the columns you need (plus partition cols)
# cols = [
#     "SID", "step", "SST","lat_left", "lon_left", "datetime", "final_wind_speed", "basin",
#     "seed", "year"
# ]

def _scan_to_df(filt):
    scanner = pds.Scanner.from_dataset(
        dataset, filter=filt, use_threads=True, batch_size=1<<18
    )
    tbl = scanner.to_table()
    df = tbl.to_pandas(types_mapper=pd.ArrowDtype)  # fast, preserves dtypes
    df = df.reset_index()
    #Filter out some tracks:
    df = df.sort_values(["SID", "datetime"], kind="mergesort").copy()

    #Condition on max wind
    max_wind_df = df.groupby("SID")["final_wind_speed"].max()
    # filtering out storms with wind speed inferior to threshold
    tracks_to_keep = max_wind_df[(max_wind_df >= 32.92)&(max_wind_df <= 100)].index
    df_return = df.loc[df["SID"].isin(tracks_to_keep)]


    return df_return

# # Read the three slices
tracks_ssp585_2025_2050 = _scan_to_df(filt_2025_2050)
tracks_ssp585_2050_2075 = _scan_to_df(filt_2050_2075)
tracks_ssp585_2075_2100 = _scan_to_df(filt_2075_2100)

tracks_ssp585_2025_2050 = tracks_ssp585_2025_2050.loc[lambda row:(~row['final_wind_speed'].isna())&(row['basin']!="SA")]
tracks_ssp585_2050_2075 = tracks_ssp585_2050_2075.loc[lambda row:~row['final_wind_speed'].isna()&(row['basin']!="SA")]
tracks_ssp585_2075_2100 = tracks_ssp585_2075_2100.loc[lambda row:~row['final_wind_speed'].isna()&(row['basin']!="SA")]

In [ ]:
def ecdf_by_basin_multi_generic(
    ib: IBTracks,
    scenarios: dict,                     # e.g. {"Historical": gen_hist, "Short": gen_short, "Medium": gen_med, "Long": gen_long}
    seed_col: str | None = "seed",
    basin_col: str = "BASIN",
    mode: str = "per_storm_max",         # 'per_storm_max' or 'all_steps'
    q_low: float = 0.1, q_high: float = 0.9,
    n_grid: int = 400,
    basins: list[str] | None = None,
    colors: dict | None = None,          # optional: {"Historical": "tab:purple", "Short": "tab:blue", ...}
    alpha_band: float = 0.20,
    savepath: str | None = None,
):
    """
    ECDF by basin comparing IBTrACS with multiple generated scenarios (incl. 'Historical').
    scenarios: dict[label -> GeneratedTracks]
    For each scenario, plot mean ECDF across seeds with a [q_low, q_high] band.
    """

    def _series_by_mode(df, basin_name=None):
        d = df if basin_name is None else df[df[basin_col] == basin_name]
        w = pd.to_numeric(d["wind"], errors="coerce")
        if mode == "per_storm_max":
            return d.assign(w=w).groupby("SID", sort=False)["w"].max().dropna().values
        elif mode == "all_steps":
            return w.dropna().values
        else:
            raise ValueError("mode must be 'per_storm_max' or 'all_steps'")

    def _ecdf_on_grid(sample, grid):
        if sample is None or len(sample) == 0:
            return np.full_like(grid, np.nan, dtype=float)
        s = np.sort(np.asarray(sample, dtype=float))
        return np.searchsorted(s, grid, side="right") / s.size

    # Basins to plot (union across IB + all scenarios)
    ib_basins = ib.df[basin_col].dropna().unique().tolist()
    gen_basins = set(ib_basins)
    for g in scenarios.values():
        df_s = g.df
        if basin_col in df_s.columns:
            gen_basins |= set(df_s[basin_col].dropna().unique().tolist())
    if basins is None:
        basins = sorted(gen_basins)
    basins.remove('SA')

    # Common wind grid across IB + all scenarios
    pools = [pd.to_numeric(ib.df["wind"], errors="coerce")]
    for g in scenarios.values():
        pools.append(pd.to_numeric(g.df["wind"], errors="coerce"))
    all_w = pd.concat(pools, axis=0).dropna()
    if all_w.empty:
        raise ValueError("No wind data available to build ECDF grid.")
    #x_min, x_max = np.nanpercentile(all_w, [0.5, 99.5])
    x_min, x_max = 0, 100
    if not np.isfinite(x_min) or not np.isfinite(x_max) or x_min >= x_max:
        x_min, x_max = float(np.nanmin(all_w)), float(np.nanmax(all_w))
    grid = np.linspace(x_min, x_max, n_grid)

    # Layout
    n = len(basins)
    ncols = min(3, max(1, int(np.ceil(np.sqrt(n)))))
    nrows = int(np.ceil(n / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows), constrained_layout=True)
    axs = np.atleast_1d(axs).ravel()

    # Default colors if not provided
    if colors is None:
        palette = ["tab:purple","tab:blue","tab:orange","tab:green","tab:red","tab:brown","tab:pink","tab:gray","tab:olive","tab:cyan"]
        colors = {label: palette[i % len(palette)] for i, label in enumerate(scenarios.keys())}

    for i, basin in enumerate(basins):
        if basin != "SA":
            ax = axs[i]

            # IBTrACS (observational historical) ECDF
            ib_sample = _series_by_mode(ib.df, basin)
            ib_ecdf = _ecdf_on_grid(ib_sample, grid)
            if np.any(np.isfinite(ib_ecdf)):
                ax.step(grid, ib_ecdf, where="post", label="IBTrACS", linewidth=1.8, color="black")

            # Each generated scenario
            for label, gen_wrap in scenarios.items():
                df_s = gen_wrap.df
                if basin_col in df_s.columns:
                    df_s = df_s[df_s[basin_col] == basin]

                if seed_col and seed_col in df_s.columns:
                    ecdf_seed = []
                    for _, g_seed in df_s.groupby(seed_col, sort=False):
                        sample = _series_by_mode(g_seed, None)
                        ecdf_s = _ecdf_on_grid(sample, grid)
                        if np.any(np.isfinite(ecdf_s)):
                            ecdf_seed.append(ecdf_s)

                    if len(ecdf_seed) > 0:
                        M = np.vstack(ecdf_seed)
                        mean_ecdf = np.nanmean(M, axis=0)
                        lo = np.nanquantile(M, q_low, axis=0)
                        hi = np.nanquantile(M, q_high, axis=0)

                        ax.step(grid, mean_ecdf, where="post", label=f"{label} (mean)", linewidth=1.4, color=colors[label])
                        ax.fill_between(grid, lo, hi, step="post", alpha=alpha_band, color=colors[label],
                                        label=f"{label} [{int(q_low*100)}–{int(q_high*100)}%]")
                else:
                    # No seed column present: single pooled ECDF
                    sample = _series_by_mode(df_s, None)
                    ecdf_pooled = _ecdf_on_grid(sample, grid)
                    if np.any(np.isfinite(ecdf_pooled)):
                        ax.step(grid, ecdf_pooled, where="post", label=f"{label}", linewidth=1.4, color=colors[label])

        ax.set_title(f"{basin} — ECDF of {'Max Wind per Storm' if mode=='per_storm_max' else 'All-Step Wind'}")
        ax.set_xlabel("Wind speed")
        ax.set_ylabel("ECDF")
        ax.set_xlim(grid[0], grid[-1])
        ax.set_ylim(0, 1)
        ax.grid(True, alpha=0.3, linewidth=0.5)
        ax.legend(fontsize=8)

    for j in range(i+1, len(axs)):
        axs[j].set_visible(False)

    if savepath:
        fig.savefig(savepath, dpi=200)
    return fig


from typing import Iterable, Optional, Dict, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import genextreme

def extreme_value_return_levels_multi(
    ib: IBTracks,
    scenarios: dict,   # e.g. {"Historical": gen_hist, "Short": gen_short, ...}
    variable: str = "wind",            # 'wind' (max) or 'pres' (min)
    block: str = "year",
    seed_col: Optional[str] = "seed",
    T: Iterable[float] = (2,5,10,20,50,100,200),
    band: str = "quantile",            # "std" or "quantile"
    q_low: float = 0.1, q_high: float = 0.9,
    min_years_per_seed: int = 8,
    colors: Optional[Dict[str, str]] = None,  # optional per-label color mapping
    savepath: Optional[str] = None,
):
    """
    Seed-aware return level plot via block maxima + GEV fit.

    - IBTrACS: single curve (one reality).
    - Generated (baseline): fit GEV per seed on annual maxima; aggregate: mean + band.
    - Scenarios (optional): dict[label -> GeneratedTracks], each plotted as mean + band (seed-aware).

    Returns:
      fig, results where results is a dict with keys:
        - "T": array of return periods
        - "ib_fit": (c,loc,scale) or None
        - "generated": {"mean_rl","lo","hi","fits_per_seed": List[(c,loc,scale)]}
        - for each scenario label: same structure as "generated"
    """
    T = np.asarray(list(T), dtype=float)
    assert variable in ("wind", "pres"), "variable must be 'wind' or 'pres'"

    def _block_max_per_year(df, var):
        # per-storm max, then map storm to its first year, then max over storms per year
        if var == "pres":
            ser = -pd.to_numeric(df["pres"], errors="coerce")  # maxima of -pres = minima of pres
        else:
            ser = pd.to_numeric(df["wind"], errors="coerce")
        years = (pd.to_numeric(df["year"], errors="coerce")
                 if "year" in df.columns else df["datetime"].dt.year)
        per_storm = df.assign(val=ser).groupby("SID", sort=False)["val"].max()
        storm_year = df.groupby("SID", sort=False)["datetime"].min().dt.year if "datetime" in df.columns else years.groupby(df["SID"]).min()
        bm = per_storm.groupby(storm_year).max().dropna().sort_index()
        return bm

    def _fit_gev(data):
        try:
            arr = np.asarray(data, dtype=float)
            if np.sum(np.isfinite(arr)) < 3:
                return None
            return genextreme.fit(arr)  # (c, loc, scale)
        except Exception:
            return None

    def _rl_from_fit(T, fit_params):
        if fit_params is None:
            return np.full_like(T, np.nan, dtype=float)
        c, loc, scale = fit_params
        q = 1.0 - 1.0/T
        return genextreme.ppf(q, c, loc=loc, scale=scale)

    def _seed_aggregate(df, label: str):
        """Return mean RL, band (lo/hi), and list of fits per seed."""
        out = {"mean_rl": np.full_like(T, np.nan, dtype=float),
               "lo": np.full_like(T, np.nan, dtype=float),
               "hi": np.full_like(T, np.nan, dtype=float),
               "fits_per_seed": []}
        if seed_col and seed_col in df.columns:
            rl_per_seed = []
            fits = []
            for s, g_s in df.groupby(seed_col, sort=False):
                bm_s = _block_max_per_year(g_s, variable)
                if bm_s.index.nunique() < min_years_per_seed:
                    continue
                fit_s = _fit_gev(bm_s.values)
                fits.append(fit_s)
                rl_s = _rl_from_fit(T, fit_s)
                rl_per_seed.append(rl_s)
            out["fits_per_seed"] = fits

            if len(rl_per_seed):
                M = np.vstack(rl_per_seed)
                out["mean_rl"] = np.nanmean(M, axis=0)
                if band == "std":
                    std = np.nanstd(M, axis=0, ddof=1)
                    out["lo"], out["hi"] = out["mean_rl"] - std, out["mean_rl"] + std
                else:
                    out["lo"] = np.nanquantile(M, q_low, axis=0)
                    out["hi"] = np.nanquantile(M, q_high, axis=0)
        else:
            # No seed dimension -> single fit on pooled data
            bm = _block_max_per_year(df, variable)
            fit = _fit_gev(bm.values)
            out["fits_per_seed"] = [fit]
            out["mean_rl"] = _rl_from_fit(T, fit)
            out["lo"] = out["hi"] = out["mean_rl"]
        return out

    # ---- IBTrACS single reality
    ib_bm = _block_max_per_year(ib.df, variable)
    ib_fit = _fit_gev(ib_bm.values)
    ib_rl = _rl_from_fit(T, ib_fit)


    # ---- Scenarios
    scenarios = scenarios or {}
    scen_results: Dict[str, Dict[str, np.ndarray]] = {}
    for lab, wrap in scenarios.items():
        scen_results[lab] = _seed_aggregate(wrap.df.copy(), lab)

    # ---- Plot
    if colors is None:
        # Basic palette; users can override via `colors`
        base_col = "tab:orange"
        palette = ["tab:blue","tab:green","tab:red","tab:purple","tab:brown","tab:pink","tab:olive","tab:cyan","tab:gray"]
        colors = {lab: palette[i % len(palette)] for i, lab in enumerate(scenarios.keys())}
    else:
        base_col = colors.get("Generated", "tab:orange")

    fig, ax = plt.subplots(figsize=(8.8, 5.6), constrained_layout=True)

    # IBTrACS
    if np.all(np.isfinite(ib_rl)):
        ax.plot(T, ib_rl, marker="o", label="IBTrACS", color="black", linewidth=2)
    else:
        ax.plot([], [], label="IBTrACS (insufficient data)", color="black")

    # Scenarios
    for lab, res in scen_results.items():
        col = colors.get(lab, None) or "tab:blue"
        ax.plot(T, res["mean_rl"], marker="o", label=f"{lab} (mean)", color=col)
        if not np.allclose(res["lo"], res["hi"], equal_nan=True):
            ax.fill_between(T, res["lo"], res["hi"], alpha=0.20, color=col, label=f"{lab} band")

    ax.set_xscale("log")
    ax.set_xlabel("Return period (years)")
    ylab = "Return level" if variable == "wind" else "Return level (on -pres scale)"
    ax.set_ylabel(ylab)
    ttl_var = "Max Wind" if variable == "wind" else "Min Pressure (as -pres)"
    ax.set_title(f"Extreme Value Return Levels — {ttl_var} (seed-aware)")
    ax.grid(True, which="both", alpha=0.3, linewidth=0.5)
    ax.legend(ncol=2, fontsize=9)
    if savepath:
        fig.savefig(savepath, dpi=200)

    results = {
        "T": T,
        "ib_fit": ib_fit,
    }
    results.update({lab: res for lab, res in scen_results.items()})
    return fig, results


In [ ]:
ibx      = IBTracks.from_df(ibtracs)
g_hist   = GeneratedTracks.from_df(tracks_historical_filtered)  # your baseline/historical simulation
g_short  = GeneratedTracks.from_df(tracks_ssp585_2025_2050)
g_medium = GeneratedTracks.from_df(tracks_ssp585_2050_2075)
g_long   = GeneratedTracks.from_df(tracks_ssp585_2075_2100)

In [ ]:
fig = ecdf_by_basin_multi_generic(
    ib=ibx,
    scenarios={
        "Historical": g_hist,
        "Short": g_short,
        "Medium": g_medium,
        "Long": g_long,
    },
    seed_col="seed",
    mode="all_steps",      # or "all_steps"
    q_low=0.1, q_high=0.9,
    colors={"Historical":"tab:purple","Short":"tab:blue","Medium":"tab:orange","Long":"tab:green"}
)

In [ ]:
fig = ecdf_by_basin_multi_generic(
    ib=ibx,
    scenarios={
        "Historical": g_hist,
        "Short": g_short,
        "Medium": g_medium,
        "Long": g_long,
    },
    seed_col="seed",
    mode="per_storm_max",      # or "all_steps"
    q_low=0.1, q_high=0.9,
    colors={"Historical":"tab:purple","Short":"tab:blue","Medium":"tab:orange","Long":"tab:green"}
)

In [ ]:
fig = extreme_value_return_levels_multi(
    ib=ibx,
    scenarios={
        "Historical": g_hist,
        "Short": g_short,
        "Medium": g_medium,
        "Long": g_long,
    },
    seed_col="seed",
    q_low=0.1, q_high=0.9,
    colors={"Historical":"tab:purple","Short":"tab:blue","Medium":"tab:orange","Long":"tab:green"}
)